# 1.3 模型到底在猜什麼？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：下一字對齊：問題與答案錯開一格**

同一格 logits 預測下一個字；shift 只做一次。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/shift.svg")))

**先想一想：**X 和 Y 相同時，模型會被教成什麼？

把「貓看狗」看成兩個問題：看到貓，下一字是看；看到看，下一字是狗。最後一個字沒有下一字，除非我們另外放結束標記。

**把直覺寫成數學：**$X=[s_0,\ldots,s_{n-2}]$，$Y=[s_1,\ldots,s_{n-1}]$；兩者長度相同。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
s = list("貓看狗。")
x, y = s[:-1], s[1:]
print(list(zip(x, y, strict=True)))
assert len(x) == len(y)

**如何讀結果：**逐格列出問題與答案；shift 是資料對齊，和 attention mask 是不同工作。

**只改一件事：**只把 Y 改成原序列，指出第一個錯誤訓練目標。
